# 3. Feature Engineering Pipeline\nDeep dive into pipeline structure, transformed features, and validation behavior.

In [ ]:
from pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport seaborn as sns\nfrom sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score\nfrom sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split\n\nfrom src.config.core import config\nfrom src.pipeline import pipe\nfrom src.processing.data_manager import load_dataset, load_pipeline, model_file_name, save_pipeline\n\nsns.set_theme(style='whitegrid')\nROOT = Path.cwd()\nif not (ROOT / 'src').exists():\n    ROOT = ROOT.parent

In [ ]:
data = load_dataset(config.app_config.training_data_file)\nx = data.drop(columns=[config.ml_config.target])\ny = data[config.ml_config.target].astype(int)\n\nx_train, x_valid, y_train, y_valid = train_test_split(\n    x, y,\n    test_size=config.ml_config.test_size,\n    random_state=config.ml_config.random_state,\n    stratify=y\n)\n\nprint('Train:', x_train.shape, 'Valid:', x_valid.shape)

In [ ]:
step_summary = [(i + 1, name, type(step).__name__) for i, (name, step) in enumerate(pipe.steps)]\nstep_summary

In [ ]:
pipe.fit(x_train, y_train)\npreds = pipe.predict(x_valid)\nproba = pipe.predict_proba(x_valid)[:, 1]\n\nmetrics = {\n    'accuracy': accuracy_score(y_valid, preds),\n    'precision': precision_score(y_valid, preds),\n    'recall': recall_score(y_valid, preds),\n    'f1': f1_score(y_valid, preds),\n    'roc_auc': roc_auc_score(y_valid, proba),\n}\nmetrics

In [ ]:
cm = confusion_matrix(y_valid, preds)\nplt.figure(figsize=(5, 4))\nsns.heatmap(cm, annot=True, fmt='d', cmap='Blues')\nplt.title('Validation Confusion Matrix')\nplt.xlabel('Predicted')\nplt.ylabel('Actual')\nplt.show()

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.ml_config.random_state)\ncv_auc = cross_val_score(pipe, x, y, cv=cv, scoring='roc_auc')\nprint('CV ROC AUC scores:', np.round(cv_auc, 4))\nprint('Mean CV ROC AUC :', round(cv_auc.mean(), 4))

In [ ]:
plt.figure(figsize=(7, 4))\nsns.histplot(proba[y_valid.values == 0], bins=30, stat='density', alpha=0.5, label='target=0')\nsns.histplot(proba[y_valid.values == 1], bins=30, stat='density', alpha=0.5, label='target=1')\nplt.title('Predicted Probability by True Class')\nplt.legend()\nplt.show()

In [ ]:
save_pipeline(pipe)\nreloaded = load_pipeline(model_file_name())\nreloaded_preds = reloaded.predict(x_valid.head(20))\nprint('Reloaded model prediction sample:', reloaded_preds[:10])